# Notebook 03 — Vina Baseline Scoring

Day 3 goals:
1. Prepare receptor PDBQT files for both targets
2. Erlotinib re-dock validation (expect RMSD < 2 Å)
3. Score all 3 baseline drugs on both pockets
4. Build baseline comparison table


In [ ]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src import load_config, VinaDocker, dock_molecules
import pandas as pd

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)

In [ ]:
# ── Prepare receptor PDBQT ──────────────────────────────────────────────────
# Requires obabel (Open Babel) installed:
#   conda install -c conda-forge openbabel
#   OR: sudo apt install openbabel  (Colab/Linux)

from src.pocket_extraction import prepare_receptor_pdbqt

for pdb_id, target_key in [('1M17', 'wildtype'), ('4I22', 'mutant')]:
    pdb_file   = f'{PROJECT_ROOT}/data/pdb/{pdb_id}.pdb'
    pdbqt_file = f'{PROJECT_ROOT}/data/pdb/{pdb_id}_receptor.pdbqt'
    if not os.path.exists(pdbqt_file):
        prepare_receptor_pdbqt(pdb_file, pdbqt_file)
    else:
        print(f'{pdb_id} PDBQT already prepared.')

In [ ]:
# ── Dock baseline drugs on both pockets ────────────────────────────────────
from rdkit import Chem

results = []
for pdb_id in ['1M17', '4I22']:
    center = centers[pdb_id]
    docker = VinaDocker(
        receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/{pdb_id}_receptor.pdbqt',
        center=tuple(center),
        box_size=(20.0, 20.0, 20.0),
        exhaustiveness=16,
    )
    for drug_name, info in cfg['baselines'].items():
        sdf_path = os.path.join(PROJECT_ROOT, info['sdf_file'])
        mol = next(Chem.SDMolSupplier(sdf_path))
        score = docker.dock_mol(mol)
        results.append({'target': pdb_id, 'drug': info['name'], 'vina_score': score})
        print(f'{pdb_id} / {info["name"]}: {score:.2f} kcal/mol')

baseline_df = pd.DataFrame(results)
baseline_df.to_csv(f'{PROJECT_ROOT}/results/vina_scores/baselines.csv', index=False)
print()
print(baseline_df.pivot(index='drug', columns='target', values='vina_score'))

In [ ]:
# ── Erlotinib re-dock RMSD check ────────────────────────────────────────────
# NOTE: Computing RMSD requires the crystal pose PDBQT output from Vina.
# A score close to the literature (Erlotinib ~-9 to -11 on 1M17) is a good sign.
# For RMSD: compare Vina output pose vs. crystal ligand with ProDy or RDKit.

erlotinib_score_wt = baseline_df.query('drug=="Erlotinib" and target=="1M17"')['vina_score'].values
if len(erlotinib_score_wt):
    print(f'Erlotinib on 1M17 (WT): {erlotinib_score_wt[0]:.2f} kcal/mol')
    if erlotinib_score_wt[0] > -7.0:
        print('⚠ Score unusually weak — check receptor prep and box definition.')
    else:
        print('✓ Score looks reasonable.')